In [1]:
# various import statements
import os
from dotenv import load_dotenv
import pandas as pd
import gspread
from google.oauth2.service_account import Credentials
import matplotlib.pyplot as plt 
import seaborn as sbn
from scipy import stats

In [2]:
# load in data from google sheet where my scores are stored
load_dotenv()
scope = ['https://www.googleapis.com/auth/drive']
jsonpath = os.environ.get('GOOGLE_APPLICATION_CREDENTIALS')
spreadsheet_id = '1suD1XqF5Dr5PdR3ggxxIyOo2_6vYakOnH1Afha72SR0'
creds = Credentials.from_service_account_file(jsonpath, scopes=scope)
gc = gspread.authorize(creds)
spreadsheet = gc.open_by_key(spreadsheet_id)
worksheet = spreadsheet.sheet1

In [3]:
data = worksheet.get_all_values()
score_df = pd.DataFrame(data[1:], columns=data[0])
score_df.head()

,Visit Date,Restaurant Name,City,Neighborhood,Cuisine,Sub Cuisine,Ingredient Quality,Execution,Personality,Price to Quality,Price to Portion,Sensory,Design,Vibe,Pacing,Staff
0,3/20/2026,Akahoshi Ramen,Chicago,Logan Square,Japanese,Ramen,8,9,7,8,7,8,5,7,7,8
1,3/21/2026,Cabra,Chicago,Fulton Market,Peruvian,,6,7,7,6,5,4,7,7,6,8


In [6]:
# score weighting definition

# food score weighting
ingredient_qual_weight = 0.3
execution_weight = 0.4
personality_weight = 0.3

# value score weighting
price_to_quality = 0.7
price_to_portion = 0.3

# atmosphere weighting
sensory_weight = 0.3
design_weight = 0.3
vibe_weight = 0.4

# hospitality weighting
pacing_weight = 0.3
staff_weight = 0.7

# overall score weighting
food_weight = 0.45
value_weight = 0.15
atmosphere_weight = 0.15
hospitality_weight = 0.25

In [7]:
# score calculation

score_df['Food Score'] = (
    score_df['Ingredient Quality'].astype(int) * ingredient_qual_weight +
    score_df['Execution'].astype(int) * execution_weight +
    score_df['Personality'].astype(int) * personality_weight
).round(1)

score_df['Value Score'] = (
    score_df['Price to Quality'].astype(int) * price_to_quality +
    score_df['Price to Portion'].astype(int) * price_to_portion
).round(1)

score_df['Atmosphere Score'] = (
    score_df['Sensory'].astype(int) * sensory_weight +
    score_df['Design'].astype(int) * design_weight +
    score_df['Vibe'].astype(int) * vibe_weight
).round(1)

score_df['Hospitality Score'] = (
    score_df['Pacing'].astype(int) * pacing_weight +
    score_df['Staff'].astype(int) * staff_weight
).round(1)

# overall score calculation
score_df['Overall Score'] = (
    score_df['Food Score'] * food_weight +
    score_df['Value Score'] * value_weight +
    score_df['Atmosphere Score'] * atmosphere_weight +
    score_df['Hospitality Score'] * hospitality_weight
).round(1)

score_df.head()

,Visit Date,Restaurant Name,City,Neighborhood,Cuisine,Sub Cuisine,Ingredient Quality,Execution,Personality,Price to Quality,...,Sensory,Design,Vibe,Pacing,Staff,Food Score,Value Score,Atmosphere Score,Hospitality Score,Overall Score
0,3/20/2026,Akahoshi Ramen,Chicago,Logan Square,Japanese,Ramen,8,9,7,8,...,8,5,7,7,8,8.1,7.7,6.7,7.7,7.7
1,3/21/2026,Cabra,Chicago,Fulton Market,Peruvian,,6,7,7,6,...,4,7,7,6,8,6.7,5.7,6.1,7.4,6.6
